# 03 — 幾何変換

## 目標
- `cv2.resize` でスケーリング
- `cv2.getRotationMatrix2D` + `cv2.warpAffine` で回転
- 3点対応アフィン変換
- `cv2.warpPerspective` で射影変換 (ホモグラフィ)

## 変換の階層
```
剛体変換 ⊂ 相似変換 ⊂ アフィン変換 ⊂ 射影変換
```
- アフィン変換: 平行線を保つ (2×3行列)
- 射影変換: 直線を保つが平行線は保たない (3×3行列)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import cv2
import numpy as np
from utils import DATA_DIR, load_image, show_nb

src = load_image(DATA_DIR / 'lena.png')
h, w = src.shape[:2]
print(f'src size: {w}x{h}')

In [ ]:
# リサイズ
half   = cv2.resize(src, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
double = cv2.resize(src, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_LINEAR)
print(f'half: {half.shape}, double: {double.shape}')
show_nb([('src', src), ('×0.5 INTER_AREA', half)], cols=2)

In [ ]:
# 回転 (中心を軸に 30°)
M_rot = cv2.getRotationMatrix2D((w/2, h/2), 30.0, 1.0)
rotated = cv2.warpAffine(src, M_rot, (w, h))
print(f'回転行列 M (2x3):\n{M_rot}')
show_nb([('src', src), ('rotate 30°', rotated)], cols=2)

In [ ]:
# アフィン変換 (3点対応)
src_pts = np.float32([[0,0],[w-1,0],[0,h-1]])
dst_pts = np.float32([[w*0.1,h*0.2],[w*0.9,h*0.1],[w*0.2,h*0.9]])
M_aff   = cv2.getAffineTransform(src_pts, dst_pts)
affined = cv2.warpAffine(src, M_aff, (w, h))
show_nb([('src', src), ('affine', affined)], cols=2)

In [ ]:
# 射影変換 (せん断ホモグラフィ)
H = np.array([[1.0, 0.2, 0.0],
              [0.1, 1.0, 0.0],
              [0.0, 0.0, 1.0]], dtype=np.float64)
warped = cv2.warpPerspective(src, H, (w, h))
print(f'H =\n{H}')
show_nb([('src', src), ('perspective (H)', warped)], cols=2)